In [1]:
!pip install tensorflow


[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Plant Disease Detection using PlantVillage Dataset
This notebook trains a Convolutional Neural Network (CNN) to detect various plant diseases from the PlantVillage dataset.

In [2]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import numpy as np
import os
import pathlib

print(f"TensorFlow Version: {tf.__version__}")

TensorFlow Version: 2.21.0


In [3]:
# Define Parameters
BATCH_SIZE = 32
IMG_SIZE = (256, 256) # Standard size for PlantVillage
EPOCHS = 20
CHANNELS = 3

# Define the path to the dataset
data_dir = pathlib.Path('PlantVillage')
print(f"Dataset path: {data_dir.absolute()}")

Dataset path: d:\python\5-DL\PlantVillage


In [4]:
# Load the dataset using tf.keras.utils.image_dataset_from_directory
# Split the dataset into 80% training and 20% validation
train_ds = tf.keras.utils.image_dataset_from_directory(
  data_dir,
  validation_split=0.2,
  subset="training",
  seed=123,
  image_size=IMG_SIZE,
  batch_size=BATCH_SIZE)

val_ds = tf.keras.utils.image_dataset_from_directory(
  data_dir,
  validation_split=0.2,
  subset="validation",
  seed=123,
  image_size=IMG_SIZE,
  batch_size=BATCH_SIZE)

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'PlantVillage'

In [ ]:
# Extract and visualize class names
class_names = train_ds.class_names
print("Class Categories:", class_names)
NUM_CLASSES = len(class_names)

In [ ]:
# Configure the dataset for performance using caching and prefetching
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
# Define Data Augmentation to prevent overfitting
data_augmentation = keras.Sequential(
  [
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
  ]
)

In [ ]:
# Build the CNN Model Architecture
model = models.Sequential([
  tf.keras.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], CHANNELS)),
  layers.Rescaling(1./255), # Normalize pixel values to [0, 1]
  data_augmentation,
  layers.Conv2D(32, (3, 3), activation='relu'),
  layers.MaxPooling2D(2, 2),
  layers.Conv2D(64, (3, 3), activation='relu'),
  layers.MaxPooling2D(2, 2),
  layers.Conv2D(128, (3, 3), activation='relu'),
  layers.MaxPooling2D(2, 2),
  layers.Flatten(),
  layers.Dense(256, activation='relu'),
  layers.Dropout(0.5), # Add dropout for regularization
  layers.Dense(NUM_CLASSES, activation='softmax')
])

In [ ]:
# Compile the Model
model.compile(
  optimizer='adam',
  loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
  metrics=['accuracy']
)

model.summary()

In [ ]:
# Train the Model
# Note: Training might take a while depending on your hardware.
history = model.fit(
  train_ds,
  validation_data=val_ds,
  epochs=EPOCHS
)

In [ ]:
# Plot Training & Validation Accuracy and Loss
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(EPOCHS)

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.show()

In [ ]:
# Save the trained model for future inference
model.save('plant_disease_model.keras')
print("Model saved successfully as 'plant_disease_model.keras'")